# Canine Atopic Dermatitis: IL-31/JAK-STAT Target Landscape
### Transcriptomic Analysis Supporting In Vitro Assay Development

---

**Author:** Portfolio Project — Drug Discovery / Animal Health  
**Dataset:** [GSE168109](https://www.ncbi.nlm.nih.gov/geo/query/acc.cgi?acc=GSE168109) — Canine whole-blood microarray (GPL13605, Agilent)  
**Study:** *Transcriptomic profile of PBMCs in dogs with atopic dermatitis before and after allergen-specific immunotherapy (ASIT)*  
**Groups:** Healthy controls (n=8) | AD pre-ASIT (n=7) | AD post-ASIT (n=7)  
**Relevance:** Elanco pipeline — Zenrelia (ilunocitinib, JAK1/3i) and Befrena (tirnovetmab, anti-IL-31 mAb)

---

## Scientific Context

Canine atopic dermatitis (CAD) is a **Th2-driven allergic skin disease** affecting 10-15% of dogs globally. It is the leading indication for Elanco's two newest products:

| Drug | Class | Target | Mechanism |
|------|-------|--------|-----------|
| **Zenrelia** (ilunocitinib) | Small molecule JAK inhibitor | JAK1 / JAK3 | Blocks downstream signaling from IL-4, IL-13, IL-31 receptors → reduces itch and inflammation |
| **Befrena** (tirnovetmab) | Anti-IL-31 mAb | IL-31 cytokine | Directly neutralizes IL-31, the key pruritogenic cytokine in CAD |

This analysis:
1. Characterizes the blood transcriptomic signature of CAD
2. Maps the IL-31 / JAK-STAT signaling axis at the gene expression level
3. Compares dog–human protein conservation for key therapeutic targets
4. Translates findings into prioritized in vitro assay recommendations


In [ ]:
import os, warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.image as mpimg
from IPython.display import display, Image
warnings.filterwarnings('ignore')

BASE  = os.path.abspath('..')
FIG   = os.path.join(BASE, 'results', 'figures')
TAB   = os.path.join(BASE, 'results', 'tables')
PROC  = os.path.join(BASE, 'data', 'processed')

def show(fname, figsize=(14, 7)):
    img = mpimg.imread(os.path.join(FIG, fname))
    fig, ax = plt.subplots(figsize=figsize)
    ax.imshow(img)
    ax.axis('off')
    plt.tight_layout()
    plt.show()

print('Environment ready.')

---
## 1. Data Overview

**Platform:** Agilent-021193 Canine (V2) Gene Expression Microarray (GPL13605)  
**Preprocessing pipeline:**
- Control probes removed (~1,400 probes)
- Quantile normalization across all 22 samples
- log2 transformation
- Duplicate probes collapsed to gene-level by mean
- Final: **13,789 genes × 22 samples**

In [ ]:
meta = pd.read_csv(os.path.join(PROC, 'sample_metadata.csv'))

def assign_group(row):
    t = row['title'].lower()
    if 'healthy' in t or 'control' in t:
        return 'Healthy'
    elif 'after therapy' in t:
        return 'AD_post'
    return 'AD_pre'

meta['group'] = meta.apply(assign_group, axis=1)

summary = meta.groupby('group').agg(
    n=('sample_id', 'count'),
    source=('source', 'first')
).reset_index()
display(summary)

expr = pd.read_csv(os.path.join(PROC, 'expression_gene_log2.csv'), index_col=0)
print(f'\nExpression matrix: {expr.shape[0]:,} genes x {expr.shape[1]} samples')

---
## 2. Principal Component Analysis

PCA on the full transcriptome reveals sample-level separation between groups.
PC1 typically captures the largest source of variation — in an AD study this often reflects the healthy vs. disease axis.

In [ ]:
show('01_PCA.png', figsize=(14, 6))

**Interpretation:** Healthy dogs (green) cluster separately from AD dogs (red = pre-ASIT, blue = post-ASIT), confirming transcriptomic separation by disease status. The partial overlap of post-ASIT samples with the AD-pre cluster suggests that 6 months of ASIT produces partial immune normalization but not complete resolution — consistent with clinical observations that ASIT is a long-term treatment.

---
## 3. Differential Expression: AD vs Healthy

Comparison of whole blood transcriptomes between AD dogs (pre-ASIT) and healthy controls.  
**Threshold:** nominal p < 0.05 and |log2FC| > 0.3 (appropriate for n=7-8 microarray groups).

Key pathway genes are annotated — these include the direct targets of Elanco's JAK inhibitor and anti-IL31 programs.

In [ ]:
show('02_Volcano_AD_vs_Healthy.png', figsize=(12, 7))

In [ ]:
de = pd.read_csv(os.path.join(TAB, 'DE_AD_vs_Healthy.csv'), index_col=0)
sig = de[de['significant']].sort_values('log2FC', ascending=False)
print(f'Total significant DEGs: {len(sig)}')
print(f'  Upregulated in AD:   {(sig.log2FC > 0).sum()}')
print(f'  Downregulated in AD: {(sig.log2FC < 0).sum()}')
print('\nTop 10 upregulated in AD:')
display(sig.head(10)[['log2FC','pval','padj']].round(4))
print('\nTop 10 downregulated in AD:')
display(sig.tail(10)[['log2FC','pval','padj']].round(4))

In [ ]:
show('03_Volcano_PostASIT_vs_PreASIT.png', figsize=(12, 7))

**Post-ASIT treatment response:** Genes that are significantly changed after 6 months of ASIT represent potential pharmacodynamic biomarkers — important for assay development, as these are the readouts that track drug effect.

---
## 4. IL-31 / JAK-STAT Pathway Deep-Dive

This section focuses on the specific gene sets most relevant to Elanco's therapeutic programs.

### The IL-31 Signaling Cascade (Befrena target pathway)

```
Th2 cells / Mast cells
        |
      IL-31  <──── Befrena (tirnovetmab) BLOCKS HERE
        |
  IL31RA / OSMR (receptor complex on DRG neurons & keratinocytes)
        |
      JAK1 / JAK2  <──── Zenrelia (ilunocitinib) BLOCKS HERE
        |
   STAT3 / STAT5
        |
   ITCH SENSATION + keratinocyte inflammatory response
```

### JAK-STAT Selectivity (Zenrelia target pathway)

```
IL-4  → IL-4R  → JAK1/JAK3 → STAT6    (Th2 differentiation)
IL-13 → IL-4R  → JAK1/TYK2 → STAT6    (IgE production, mucus)
IL-31 → IL31RA → JAK1/JAK2 → STAT3    (ITCH)
IL-2  → IL-2R  → JAK1/JAK3 → STAT5    (T cell survival)

Ilunocitinib = selective JAK1/JAK3 inhibitor
Concern: JAK2 inhibition → anemia (hematopoiesis)
```

In [ ]:
show('04_Pathway_Heatmap.png', figsize=(16, 14))

**Heatmap interpretation:** Each column is one dog. Columns are ordered: Healthy | AD Pre-ASIT | AD Post-ASIT. Gene expression is shown as Z-score (red = high, blue = low relative to the sample mean).

Key observations:
- **Th2 signature genes** (GATA3, IL4, IL13, CCL17) tend to be elevated in AD_pre samples
- **JAK pathway components** (JAK1, JAK2, STAT3) show moderate upregulation in AD blood
- **Post-ASIT** samples show partial normalization toward the Healthy profile for several Th2 genes
- **Pruritus mediators** (TRPV1, TRPA1) show expression in circulating cells, consistent with neuro-immune communication

In [ ]:
show('05_Pathway_GroupMeans.png', figsize=(16, 14))

In [ ]:
show('07_Pathway_FC_Lollipop.png', figsize=(10, 12))

**Fold change summary note:** Many core pathway genes (JAK1, STAT3, etc.) show modest fold changes in blood. This is biologically expected — blood is not the primary site of IL-31 production (which occurs in skin-resident Th2 cells). The **key assay implication**: in vitro assays should use **stimulated** PBMC conditions (anti-CD3/CD28 + IL-4/IL-31 challenge) to amplify pathway activation and create a pharmacologically relevant window for compound testing.

In [ ]:
show('08_Disease_vs_Treatment_Dynamics.png', figsize=(10, 9))

---
## 5. Canine–Human Gene Conservation Analysis

A critical question for assay development: **can we use human cell lines and recombinant proteins as surrogates for canine targets?**

This matters because:
- Most commercial reagents (antibodies, recombinant proteins, cell lines) are designed for human proteins
- High amino acid identity (>85%) generally means human tools are valid surrogates
- Low identity (<60%) means species-specific reagents are required

In [ ]:
show('06_Canine_Human_Conservation.png', figsize=(11, 10))

In [ ]:
cons = pd.read_csv(os.path.join(TAB, 'canine_human_conservation.csv'))
display(cons[['Gene','AA_Identity_pct','Assay_Priority','Biological_Relevance']].to_string(index=False))

### Key Findings on Conservation:

| Finding | Implication |
|---------|-------------|
| JAK1 (97%), JAK2 (96%), STAT3 (99%), STAT6 (96%) are nearly identical | Human recombinant JAK proteins are valid for biochemical kinase assays; human cell lines valid for JAK selectivity profiling |
| IL31RA (76%), OSMR (82%) are moderately conserved | Human cells can provide a partial signal, but a canine-specific BaF3 reporter line is preferred for Befrena potency/neutralization assays |
| IL-4 (42%), IL-13 (48%) are poorly conserved | Cannot rely on human IL-4/IL-13 to stimulate canine cells; must use canine-specific recombinant cytokines |
| FLG (30%) is extremely divergent | Filaggrin-based barrier assays developed for human AD cannot be directly translated to dogs |
| GATA3 (99%) is essentially identical | Human Th2 reporter cells (GATA3-GFP, etc.) are valid for mechanistic studies |

---
## 6. In Vitro Assay Recommendations

Based on the transcriptomic findings and conservation analysis, the following cell-based assay cascade is recommended for supporting Elanco's CAD drug discovery portfolio.

In [ ]:
show('09_Assay_Priority_Matrix.png', figsize=(14, 7))

In [ ]:
assays = pd.read_csv(os.path.join(TAB, 'assay_recommendations.csv'))
pd.set_option('display.max_colwidth', 80)
display(assays[['Priority','Assay_Name','Target','Cell_Model','Readout','Feasibility','Relevant_Drug']]
        .sort_values('Priority').to_string(index=False))

### Proposed Screening Cascade

```
┌─────────────────────────────────────────────────────────────────────┐
│           ELANCO CAD IN VITRO SCREENING CASCADE                     │
├─────────────────────────────────────────────────────────────────────┤
│                                                                     │
│  TIER 1 — PRIMARY SCREENING (High throughput)                       │
│  ┌────────────────────────────────────────────────────────┐         │
│  │ JAK1 Biochemical Kinase Assay (ADP-Glo/TR-FRET)       │         │
│  │  • IC50 determination against recombinant JAK1/3      │         │
│  │  • Selectivity vs JAK2 and TYK2                       │         │
│  │  • Human protein valid surrogate (97% identity)       │         │
│  └────────────────────────────────────────────────────────┘         │
│                              |                                      │
│                              v  (hits with >10x JAK1/JAK2)         │
│  TIER 2 — CELLULAR CONFIRMATION (Medium throughput)                 │
│  ┌────────────────────────────────────────────────────────┐         │
│  │ STAT6 pY641 assay in DH82 canine macrophages          │         │
│  │  • IL-4 stimulation → pSTAT6 flow cytometry           │         │
│  │  • Confirms cell permeability + JAK1 engagement       │         │
│  │  • DH82 is ATCC-sourced canine macrophage line        │         │
│  └────────────────────────────────────────────────────────┘         │
│                              |                                      │
│                              v  (cellular IC50 < 1 uM)             │
│  TIER 3 — MECHANISM / BIOLOGIC TESTING (Low throughput)             │
│  ┌────────────────────────────────────────────────────────┐         │
│  │ Canine PBMC cytokine release panel                    │         │
│  │  • Anti-CD3/CD28 + recombinant canine IL-4 stimulus   │         │
│  │  • Multiplex: IL-31, IL-4, IL-13, IFNg (MSD panel)   │         │
│  │  • Tests physiologically relevant Th2 biology         │         │
│  ├────────────────────────────────────────────────────────┤         │
│  │ IL-31 Neutralization Bioassay (Befrena QC / SAR)     │         │
│  │  • BaF3 + canine IL31RA/OSMR stable line             │         │
│  │  • IL-31 driven proliferation (CTG readout)           │         │
│  │  • IC50 of anti-IL31 mAb neutralization              │         │
│  └────────────────────────────────────────────────────────┘         │
│                                                                     │
└─────────────────────────────────────────────────────────────────────┘
```

---
## 7. Cell Line Models for Canine AD Research

The table below summarizes available canine cell models relevant to CAD assay development:

In [ ]:
cell_lines = pd.DataFrame([
    {
        'Cell Line': 'DH82',
        'Species': 'Canis lupus familiaris',
        'Cell Type': 'Macrophage (histiocytic sarcoma)',
        'Catalog': 'ATCC CRL-10389',
        'Key Features': 'Canine macrophage; expresses JAK1, STAT6; responds to IL-4; used for JAK inhibitor testing',
        'Limitations': 'Tumor cell line; not primary; low IL-31R expression',
        'Assay Use': 'pSTAT6 assay, JAK1 cellular confirmation'
    },
    {
        'Cell Line': 'Canine PBMCs (primary)',
        'Species': 'Canis lupus familiaris',
        'Cell Type': 'Mixed lymphocytes/monocytes',
        'Catalog': 'Bioreclamation IVT / ATCC',
        'Key Features': 'Most physiologically relevant; responds to anti-CD3/CD28; produces IL-31, IL-4 when Th2-skewed',
        'Limitations': 'Donor variability; short-lived; not amenable to genetic manipulation',
        'Assay Use': 'Cytokine release panel; primary pharmacology confirmation'
    },
    {
        'Cell Line': 'CPEK (Canine Primary Epidermal Keratinocytes)',
        'Species': 'Canis lupus familiaris',
        'Cell Type': 'Epidermal keratinocyte',
        'Catalog': 'Commercial suppliers (Cell Biologics)',
        'Key Features': 'Primary keratinocytes; expresses IL31RA, OSMR; barrier function measurable (TEER); S100A8 secretion',
        'Limitations': 'Limited passages; requires specialized media; expensive per lot',
        'Assay Use': 'Barrier disruption assay; IL-31 downstream signaling (pSTAT3)'
    },
    {
        'Cell Line': 'BaF3 + canine IL31RA/OSMR (engineered)',
        'Species': 'Mus musculus (parental) + canine transgene',
        'Cell Type': 'IL-3-dependent pro-B cell',
        'Catalog': 'Engineered in-house (standard protocol)',
        'Key Features': 'Proliferation depends on canine IL-31 upon IL-3 withdrawal; gold standard for mAb neutralization IC50',
        'Limitations': 'Requires generation and validation; murine background',
        'Assay Use': 'Befrena potency/neutralization; IL-31 mAb SAR during discovery'
    },
    {
        'Cell Line': 'HEK293 + canine IL31RA/OSMR-STAT3-luc',
        'Species': 'Homo sapiens (parental) + canine transgene',
        'Cell Type': 'Kidney epithelial (reporter)',
        'Catalog': 'Engineered in-house',
        'Key Features': 'Luciferase reporter driven by STAT3 response elements; quantitative, high-throughput compatible',
        'Limitations': 'Artificial overexpression; background STAT3 activity',
        'Assay Use': 'IL-31 → JAK → STAT3 pathway assay; high-throughput screening'
    },
])

display(cell_lines)

---
## 8. Summary & Key Insights

### What the data tells us:

1. **CAD dogs have a measurable whole-blood transcriptomic signature** — 260 DEGs separate AD from healthy controls, even in blood (not skin), demonstrating systemic immune activation

2. **The Th2/JAK/STAT axis is activated in canine AD blood** — GATA3, IL4, IL13, STAT3, JAK1, JAK2 all show directional upregulation in AD, consistent with Th2 polarization

3. **IL-31 itself is not reliably detected in blood microarrays** — consistent with published literature noting that IL-31 mRNA is unstable/transient; protein ELISA (Befrena biomarker studies) or stimulated-PBMC assays are needed

4. **ASIT partially reverses the transcriptomic signature** — 80 genes change significantly after 6 months, providing pharmacodynamic biomarker candidates for Elanco's immunotherapy programs

5. **JAK pathway genes are >91% conserved between dogs and humans** — human recombinant proteins and cell lines are valid biochemical surrogates for primary JAK inhibitor screening

6. **IL-31/IL31RA are 74-76% conserved** — species-specific validation is required for Befrena-class biologics; canine-specific cell line tools (BaF3 + canine IL31RA) should be priority investments

### Proposed immediate assay investments:

| Priority | Assay | Timeline | Strategic Value |
|----------|-------|----------|-----------------|
| 1 | JAK1/3 biochemical kinase panel | Weeks 1-4 | Screen all small molecule scaffolds; JAK2 selectivity flag |
| 1 | pSTAT6 assay in DH82 cells | Weeks 2-6 | Cellular confirmation; cell penetration verification |
| 1 | Canine PBMC cytokine multiplex | Weeks 4-8 | Physiological relevance; Th2 functional readout |
| 2 | BaF3/canine IL31RA neutralization assay | Months 2-4 | Critical for biologic MOA confirmation and QC |
| 2 | CPEK barrier disruption assay | Months 3-6 | Novel readout for combination small mol + biologic |

---
## References

1. **GSE168109** — Transcriptomic profile of PBMCs in canine AD before/after ASIT. *Immunogenetics* (2023). [GEO](https://www.ncbi.nlm.nih.gov/geo/query/acc.cgi?acc=GSE168109)
2. Elanco Animal Health — [Befrena (tirnovetmab) USDA Approval](https://www.elanco.com/us/newsroom/press-releases/befrena-usda-approval), December 2025
3. Elanco Animal Health — [Zenrelia (ilunocitinib) Monograph 2026](https://pro.dermavet.com/zenrelia-ilunocitinib-in-dogs-2026-monograph/)
4. Gonzales AJ et al. (2021) Oclacitinib (APOQUEL) is a novel Janus kinase inhibitor with activity against cytokines involved in allergy. *J Vet Pharmacol Ther*.
5. Marsella R & Girolomoni G (2009) Canine models of atopic dermatitis: a useful tool with great potential. *J Invest Dermatol*.
6. Elanco Investor Day 2025 — [R&D Pipeline Update](https://www.elanco.com/us/newsroom/press-releases/elanco-investor-day-2025)
7. Schlotter YM et al. (2011) Lesional skin in atopic dogs shows a mixed Type-1 and Type-2 immune responsiveness. *Vet Immunol Immunopathol*.
8. Nomura I et al. (2003) Cytokine milieu of atopic dermatitis: comparative genomics. *J Allergy Clin Immunol*.

---
*Analysis performed with Python 3.10, GEOparse, pandas, scikit-learn, matplotlib, seaborn.*  
*All code available in `scripts/` directory. Data: NCBI GEO public domain.*